# 🔧 Preprocessing — Analyse de Sentiment Bancaire
**Objectif :** Nettoyer et préparer le texte pour la modélisation.

In [1]:
# 📦 Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Télécharger les ressources NLTK
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

sns.set_theme(style="whitegrid")
print("✅ Bibliothèques importées !")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\emman\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\emman\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\emman\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\emman\AppData\Roaming\nltk_data...


✅ Bibliothèques importées !


[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\emman\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# 📂 Chargement
df = pd.read_csv(
    r'C:\Users\emman\Documents\Portfolio\sentiment-analysis-banking\data\data.csv'
)
print(f"✅ Dataset chargé : {df.shape[0]:,} lignes")
print(f"\n👀 Exemple de texte brut :")
print(df['Sentence'].iloc[0])

✅ Dataset chargé : 5,842 lignes

👀 Exemple de texte brut :
The GeoSolutions technology will leverage Benefon 's GPS solutions by providing Location Based Search Technology , a Communities Platform , location relevant multimedia content and a new and powerful commercial model .


In [3]:
# 🧹 Fonction de nettoyage du texte
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def nettoyer_texte(texte):
    # Étape 1 : Mettre en minuscules
    texte = texte.lower()
    
    # Étape 2 : Supprimer les URLs
    texte = re.sub(r'http\S+|www\S+', '', texte)
    
    # Étape 3 : Supprimer les chiffres
    texte = re.sub(r'\d+', '', texte)
    
    # Étape 4 : Supprimer la ponctuation
    texte = re.sub(r'[^a-zA-Z\s]', '', texte)
    
    # Étape 5 : Supprimer les espaces multiples
    texte = re.sub(r'\s+', ' ', texte).strip()
    
    # Étape 6 : Tokeniser (découper en mots)
    mots = word_tokenize(texte)
    
    # Étape 7 : Supprimer stopwords + Lemmatiser
    mots_nettoyes = [
        lemmatizer.lemmatize(mot)
        for mot in mots
        if mot not in stop_words
        and len(mot) > 2
    ]
    
    # Étape 8 : Recoller les mots
    return ' '.join(mots_nettoyes)

# Test sur un exemple
exemple = df['Sentence'].iloc[0]
print("📝 AVANT nettoyage :")
print(exemple)
print(f"\n✅ APRÈS nettoyage :")
print(nettoyer_texte(exemple))

📝 AVANT nettoyage :
The GeoSolutions technology will leverage Benefon 's GPS solutions by providing Location Based Search Technology , a Communities Platform , location relevant multimedia content and a new and powerful commercial model .

✅ APRÈS nettoyage :
geosolutions technology leverage benefon gps solution providing location based search technology community platform location relevant multimedia content new powerful commercial model


In [4]:
# 🧹 Appliquer le nettoyage sur tout le dataset
print("⏳ Nettoyage en cours...")

df['cleaned_text'] = df['Sentence'].apply(nettoyer_texte)

print("✅ Nettoyage terminé !")
print(f"\n📊 Aperçu des résultats :")
print(df[['Sentence', 'cleaned_text', 'Sentiment']].head())

# Vérifier les textes vides après nettoyage
textes_vides = df['cleaned_text'].str.strip().eq('').sum()
print(f"\n⚠️ Textes vides après nettoyage : {textes_vides}")

⏳ Nettoyage en cours...
✅ Nettoyage terminé !

📊 Aperçu des résultats :
                                            Sentence  \
0  The GeoSolutions technology will leverage Bene...   
1  $ESI on lows, down $1.50 to $2.50 BK a real po...   
2  For the last quarter of 2010 , Componenta 's n...   
3  According to the Finnish-Russian Chamber of Co...   
4  The Swedish buyout firm has sold its remaining...   

                                        cleaned_text Sentiment  
0  geosolutions technology leverage benefon gps s...  positive  
1                           esi low real possibility  negative  
2  last quarter componenta net sale doubled eurm ...  positive  
3  according finnishrussian chamber commerce majo...   neutral  
4  swedish buyout firm sold remaining percent sta...   neutral  

⚠️ Textes vides après nettoyage : 1


In [5]:
# 🗑️ Supprimer les textes vides
print(f"📊 Taille avant nettoyage : {len(df)}")

# Trouver et afficher le texte vide
texte_vide = df[df['cleaned_text'].str.strip() == '']
print(f"\n⚠️ Texte vide trouvé :")
print(texte_vide[['Sentence', 'Sentiment']])

# Supprimer les textes vides
df = df[df['cleaned_text'].str.strip() != '']
df = df.reset_index(drop=True)

print(f"\n📊 Taille après suppression : {len(df)}")
print(f"✅ {1} ligne supprimée !")

📊 Taille avant nettoyage : 5842

⚠️ Texte vide trouvé :
        Sentence Sentiment
921  It 's not .   neutral

📊 Taille après suppression : 5841
✅ 1 ligne supprimée !


In [6]:
# 🔢 Encoder les sentiments en chiffres
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['Sentiment'])

print("✅ Encodage effectué !")
print(f"\n📊 Correspondance :")
for sentiment, code in zip(
    le.classes_,
    le.transform(le.classes_)
):
    print(f"   {sentiment} → {code}")

print(f"\n📊 Distribution des labels :")
print(df['label'].value_counts())

✅ Encodage effectué !

📊 Correspondance :
   negative → 0
   neutral → 1
   positive → 2

📊 Distribution des labels :
label
1    3129
2    1852
0     860
Name: count, dtype: int64


In [7]:
# Sauvegarder le dataset nettoyé
df.to_csv(
    r'C:\Users\emman\Documents\Portfolio\sentiment-analysis-banking\data\data_cleaned.csv',
    index=False
)

print("Dataset nettoyé sauvegardé !")
print(f"Taille finale : {len(df)} lignes")
print(f"Colonnes : {list(df.columns)}")

Dataset nettoyé sauvegardé !
Taille finale : 5841 lignes
Colonnes : ['Sentence', 'Sentiment', 'cleaned_text', 'label']
